# D19 — Infraestrutura e validação do estimador DiD escalonado

## tl;dr

A infraestrutura científica anterior à primeira estimação real foi construída e validada até o limite permitido pelo ambiente. O DGP sintético é determinístico, produz a verdade `ATT(g,t)` conhecida e cobre heterogeneidade, violação de tendências paralelas, overlap fraco, antecipação e coorte `n=2`. O dry-run da D15 valida 129 tratados, 4.963 controles, coortes, máscaras, chaves e event-time sem chamar qualquer estimador.

**Bloqueio objetivo:** não existe backend Callaway–Sant'Anna instalado. R/Rscript e o pacote `did` estão ausentes; o pacote Python `differences` também está ausente. `statsmodels` não calcula group-time ATT e é usado somente para uma demonstração TWFE sintética. Nenhuma aproximação caseira recebe o nome Callaway–Sant'Anna.

Consequentemente, `D19_BACKEND_CALLAWAY_SANTANNA_VALIDADO = BLOQUEADO_DEPENDENCIA` e `D19_INFRAESTRUTURA_PRONTA_PARA_ESTIMACAO_REAL = NAO`. Nenhum efeito real foi estimado.

## Contexto, método e fronteiras

A D19 vem depois da reavaliação D18 e antes da primeira estimação causal. A unidade real continua município-ano, 2007–2019; o outcome principal é `pessoal_ocupado_assalariado` em nível; o controle principal é never-treated; `anticipation=0`; e `k=-1` é a referência dinâmica.

A base real é usada exclusivamente para validação estrutural e montagem do formato futuro. Todas as estimações existentes neste notebook usam dados com `fonte_dados='SINTETICO_D19'`. Não se calcula ATT real, ATT(g,t) real, event-study causal real, TWFE real, p-valor ou intervalo causal da política.

In [1]:
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import estima_did_escalonado as infraestrutura
import simula_did_escalonado as simulador
from visualizacao_ipt import aplicar_tema_ipt, salvar_figura_ipt, estilizar_tabela_ipt, CORES_IPT

PATH_AMOSTRA = ROOT / 'data' / 'processed' / 'amostra_causal_cempre_2007_2019.parquet'
OUT_DIAG = ROOT / 'outputs' / 'diagnostics'
OUT_FIG = ROOT / 'outputs' / 'figures'

def mostrar(frame, n=30):
    display(estilizar_tabela_ipt(frame.head(n)))

def salvar_figura(fig, nome):
    aplicar_tema_ipt(fig)
    caminho = salvar_figura_ipt(fig, OUT_FIG / f'{nome}.png')
    print(f'Figura preservada em: {caminho.relative_to(ROOT)}')
    return fig

def git(*args):
    return subprocess.run(['git', *args], cwd=ROOT, check=True, capture_output=True, text=True).stdout.strip()

## 1. Auditoria do ambiente e backends

In [2]:
estado_git_inicial = {
    'branch': git('branch', '--show-current'),
    'head': git('rev-parse', 'HEAD'),
    'origin_main': git('rev-parse', 'origin/main'),
    'status_projeto': git('status', '--short', '--', '.'),
}
backends = infraestrutura.auditar_backends()
decisao_backend = infraestrutura.selecionar_backend(backends)
backends.to_csv(OUT_DIAG / 'D19_backends.csv', index=False, encoding='utf-8-sig')
display(pd.DataFrame({'campo': estado_git_inicial.keys(), 'valor': estado_git_inicial.values()}))
mostrar(backends, 10)
display(pd.DataFrame([decisao_backend]))

,campo,valor
0,branch,main
1,head,c59022fc15e081411ff09bdf8765bc6347d15b00
2,origin_main,c59022fc15e081411ff09bdf8765bc6347d15b00
3,status_projeto,M README.md\n M docs/playbooks/ESTADO_ATUAL.md...


,backend,linguagem,versao,disponivel_no_ambiente,suporta_group_time_ATT,suporta_never_treated,suporta_anticipation,suporta_event_study,suporta_covariaveis,suporta_inferencia_cluster,observacao
0,did,R,AUSENTE__REFERENCIA_DOCUMENTAL_2.5.1,False,SIM,SIM,SIM,SIM__aggte_dynamic,SIM,SIM,Implementação canônica dos autores; att_gt/aggte. R/Rscript e pacote did não estão disponíveis neste ambiente.
1,differences,Python,AUSENTE__REFERENCIA_DOCUMENTAL_0.3.0,False,SIM,SIM,NAO_VALIDADO_LOCALMENTE,SIM__aggregate_event,SIM,NAO_VALIDADO_LOCALMENTE,"ATTgt declara compatibilidade com Callaway–Sant'Anna, mas o pacote não está instalado e sua semântica completa não foi validada localmente."
2,statsmodels,Python,0.14.6,True,NAO,NAO_COMO_CS,NAO_COMO_CS,NAO_COMO_CS,SIM,SIM__REGRESSAO,Disponível apenas para demonstração TWFE em dados sintéticos; não é backend Callaway–Sant'Anna.


,gate,backend_selecionado,usar_aproximacao_caseira,recomendacao
0,BLOQUEADO_DEPENDENCIA,None,False,Disponibilizar R + did==2.5.1 ou aprovar audit...


**Decisão.** O backend primário recomendado é o pacote R `did` dos autores, versão documental 2.5.1, mas ele não está disponível porque não há R/Rscript local. `differences` 0.3.0 é um candidato Python compatível com ATTgt, porém também está ausente e não teve antecipação/clustering validados localmente. `statsmodels` 0.15.0 está instalado, mas não implementa `ATT(g,t)` de Callaway–Sant'Anna. O gate fica bloqueado por dependência, sem fallback caseiro.

## 2. Contrato do estimador e mapeamento metodológico

In [3]:
contrato = infraestrutura.contrato_estimador_d19()
mapeamento = infraestrutura.mapeamento_backend_did_r()
contrato.to_csv(OUT_DIAG / 'D19_contrato_estimador.csv', index=False, encoding='utf-8-sig')
mapeamento.to_csv(OUT_DIAG / 'D19_mapeamento_backend.csv', index=False, encoding='utf-8-sig')
mostrar(contrato, 30)
mostrar(mapeamento, 30)

,campo,valor,status
0,id,codigo_municipio_ibge,CONTRATO_D19_SEM_ESTIMACAO_REAL
1,tempo,ano,CONTRATO_D19_SEM_ESTIMACAO_REAL
2,grupo_coorte,coorte_g; 0 no backend para never-treated,CONTRATO_D19_SEM_ESTIMACAO_REAL
3,controle,never-treated,CONTRATO_D19_SEM_ESTIMACAO_REAL
4,outcome,pessoal_ocupado_assalariado,CONTRATO_D19_SEM_ESTIMACAO_REAL
5,anticipation,0,CONTRATO_D19_SEM_ESTIMACAO_REAL
6,periodo_estimacao,2007-2019_INTEGRAL,CONTRATO_D19_SEM_ESTIMACAO_REAL
7,mascara,elegivel_estimacao_principal,CONTRATO_D19_SEM_ESTIMACAO_REAL
8,tratados_populacao_principal,129,CONTRATO_D19_SEM_ESTIMACAO_REAL
9,controles,4963,CONTRATO_D19_SEM_ESTIMACAO_REAL


,conceito_artigo,parametro_backend,valor_projeto,justificativa
0,unidade,idname,codigo_municipio_ibge,chave municipal congelada
1,período,tname,ano,painel anual 2007-2019
2,first treatment,gname,g_backend,coorte; zero para never-treated
3,outcome,yname,pessoal_ocupado_assalariado,outcome primário D14
4,painel,panel,TRUE,município-ano longitudinal
5,painel balanceado,allow_unbalanced_panel,FALSE,contrato principal; máscara Cabo Frio deve ser tratada explicitamente
6,controle,control_group,nevertreated,grupo principal congelado
7,antecipação,anticipation,0,hipótese identificadora congelada
8,covariáveis,xformla,~1,principal incondicional
9,método,est_method,dr,default oficial; intercepto apenas no principal


O painel completo 2007–2019 é o período planejado para identificação/estimação; `k=-2...+2` é somente a janela principal de apresentação. No backend R `did`, `base_period='universal'` representa a normalização de reporte em `g-1`, enquanto a janela do gráfico será aplicada somente depois da estimação. O painel nunca deve ser truncado silenciosamente à janela visual.

## 3. DGP sintético reproduzível

In [4]:
cenarios = {
    'LIMPO': simulador.criar_cenario_limpo(seed=1901, n_unidades=240),
    'HETEROGENEIDADE_FORTE': simulador.criar_cenario_heterogeneo(seed=1902, n_unidades=240),
    'VIOLACAO_PARALLEL_TRENDS': simulador.criar_cenario_violacao_tendencias(seed=1903, n_unidades=240),
    'OVERLAP_FRACO': simulador.criar_cenario_overlap_fraco(seed=1904, n_unidades=240),
    'COORTE_2013_N2': simulador.criar_cenario_coorte_pequena(seed=1905, n_unidades=240),
}
resumo_cenarios = pd.DataFrame([resultado.metadados for resultado in cenarios.values()])
verdade_att_gt = pd.concat([resultado.verdade_att_gt for resultado in cenarios.values()], ignore_index=True)
verdade_event_time = pd.concat([resultado.verdade_event_time for resultado in cenarios.values()], ignore_index=True)
resumo_cenarios.to_csv(OUT_DIAG / 'D19_cenarios_sinteticos.csv', index=False, encoding='utf-8-sig')
verdade_att_gt.to_csv(OUT_DIAG / 'D19_att_gt_verdade.csv', index=False, encoding='utf-8-sig')
verdade_event_time.to_csv(OUT_DIAG / 'D19_event_study_verdade.csv', index=False, encoding='utf-8-sig')
mostrar(resumo_cenarios, 10)

,cenario,seed,n_unidades,n_never_treated,n_tratados,contagem_coortes,parallel_trends_verdadeira,overlap_fraco,anticipation_periodos
0,LIMPO,1901,240,96,144,"{2010: 48, 2012: 48, 2013: 48}",True,False,0
1,HETEROGENEIDADE_FORTE,1902,240,96,144,"{2010: 48, 2012: 48, 2013: 48}",True,False,0
2,VIOLACAO_PARALLEL_TRENDS,1903,240,96,144,"{2010: 48, 2012: 48, 2013: 48}",False,False,0
3,OVERLAP_FRACO,1904,240,96,144,"{2010: 48, 2012: 48, 2013: 48}",True,True,0
4,COORTE_2013_N2,1905,240,96,144,"{2010: 71, 2012: 71, 2013: 2}",True,False,0


O DGP controla efeitos fixos individuais e temporais, tendência comum, ruído, efeitos por coorte, dinâmica pós-tratamento, antecipação, violação de tendências paralelas, overlap fraco e tamanho de coorte. A seed é fixa. `y0_sem_tratamento` e `efeito_verdadeiro` existem somente porque os dados são sintéticos.

## 4. O que significa ATT(g,t)

`ATT(g,t)` é o efeito médio para as unidades cuja primeira exposição ocorreu em `g`, avaliado no período `t`. Em linguagem natural, `ATT(2011, 2013)` pergunta qual é, em 2013, o efeito médio para a coorte que começou o tratamento em 2011. O event-time correspondente é `t-g=2`.

In [5]:
exemplo_att = cenarios['LIMPO'].verdade_att_gt.query('coorte_g == 2010 and ano == 2013')
mostrar(exemplo_att, 5)
display(Markdown('A coluna `att_gt_verdadeiro` é parâmetro conhecido do DGP, não estimativa.'))

,coorte_g,ano,att_gt_verdadeiro,n_tratados_coorte,event_time,natureza,cenario
3,2010,2013,5.000000,48,3,VERDADE_DGP_NAO_ESTIMATIVA,LIMPO


A coluna `att_gt_verdadeiro` é parâmetro conhecido do DGP, não estimativa.

## 5. Cenário limpo e validação bloqueada do backend

In [6]:
limpo = cenarios['LIMPO']
trajetorias_limpas = limpo.painel.assign(grupo=lambda d: np.where(d['coorte_g'].eq(0), 'Never-treated', 'g=' + d['coorte_g'].astype(str))).groupby(['grupo', 'ano'], as_index=False)['y'].mean()
fig = px.line(trajetorias_limpas, x='ano', y='y', color='grupo', markers=True, labels={'ano': 'Ano', 'y': 'Outcome sintético médio', 'grupo': 'Grupo'}, title='Cenário limpo: adoção escalonada com tendências paralelas no DGP')
salvar_figura(fig, 'd19_01_cenario_limpo_trajetorias')
fig.show()
validacao_backend_limpo = pd.DataFrame([
    {'metrica': 'bias_ATT_gt', 'valor': np.nan, 'status': 'NAO_CALCULADO_BACKEND_AUSENTE'},
    {'metrica': 'RMSE_ATT_gt', 'valor': np.nan, 'status': 'NAO_CALCULADO_BACKEND_AUSENTE'},
    {'metrica': 'cobertura_intervalos', 'valor': np.nan, 'status': 'NAO_CALCULADO_BACKEND_AUSENTE'},
])
mostrar(validacao_backend_limpo, 10)

Figura preservada em: outputs\figures\interactive\d19_01_cenario_limpo_trajetorias.html


,metrica,valor,status
0,bias_ATT_gt,nan,NAO_CALCULADO_BACKEND_AUSENTE
1,RMSE_ATT_gt,nan,NAO_CALCULADO_BACKEND_AUSENTE
2,cobertura_intervalos,nan,NAO_CALCULADO_BACKEND_AUSENTE


O cenário limpo satisfaz tendências paralelas, `anticipation=0` e overlap adequado. A recuperação de `ATT(g,t)`, bias, RMSE e cobertura não é declarada: faltou o backend confiável. Preencher essas células com um estimador artesanal violaria o objetivo da D19.

## 6. Heterogeneidade e comparação TWFE exclusivamente sintética

In [7]:
heterogeneo = cenarios['HETEROGENEIDADE_FORTE']
twfe = infraestrutura.estimar_twfe_sintetico(heterogeneo.painel)
verdade_global = simulador.agregar_verdade_att(heterogeneo.verdade_att_gt, 'global').iloc[0]['att_verdadeiro_agregado']
comparacao_twfe = pd.DataFrame([
    {'objeto': 'Média global da verdade ATT(g,t)', 'valor': verdade_global, 'natureza': 'VERDADE_DGP'},
    {'objeto': 'Coeficiente TWFE convencional', 'valor': twfe['coeficiente_twfe'], 'natureza': 'ESTIMATIVA_SINTETICA_NAO_CS'},
])
comparacao_twfe.to_csv(OUT_DIAG / 'D19_twfe_sintetico.csv', index=False, encoding='utf-8-sig')
mostrar(comparacao_twfe, 10)
fig = px.bar(comparacao_twfe, x='objeto', y='valor', color='natureza', text='valor', labels={'objeto': '', 'valor': 'Valor sintético', 'natureza': 'Natureza'}, title='Heterogeneidade: TWFE não é o mesmo objeto que a agregação de ATT(g,t)')
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
salvar_figura(fig, 'd19_02_twfe_vs_verdade_sintetica')
fig.show()

,objeto,valor,natureza
0,"Média global da verdade ATT(g,t)",11.356367,VERDADE_DGP
1,Coeficiente TWFE convencional,10.406507,ESTIMATIVA_SINTETICA_NAO_CS


Figura preservada em: outputs\figures\interactive\d19_02_twfe_vs_verdade_sintetica.html


A comparação é didática e não afirma que o coeficiente TWFE seja estimativa Callaway–Sant'Anna. Com efeitos heterogêneos e timing variável, ele combina comparações e horizontes em um único número cuja pergunta não coincide com as agregações modernas de `ATT(g,t)`.

## 7. Violação deliberada de parallel trends

In [8]:
violacao = cenarios['VIOLACAO_PARALLEL_TRENDS']
diagnostico_violacao = simulador.diagnosticar_tendencias_nao_tratadas(violacao.painel)
display(pd.DataFrame([diagnostico_violacao]))
pre_primeira_coorte = violacao.painel.loc[violacao.painel['ano'].lt(min(simulador.ConfiguracaoDGP().coortes))].assign(grupo=lambda d: np.where(d['coorte_g'].eq(0), 'Never-treated', 'Futuros tratados')).groupby(['grupo', 'ano'], as_index=False)['y0_sem_tratamento'].mean()
fig = px.line(pre_primeira_coorte, x='ano', y='y0_sem_tratamento', color='grupo', markers=True, labels={'ano': 'Ano pré', 'y0_sem_tratamento': 'Y(0) médio conhecido', 'grupo': 'Grupo'}, title='Violação construída: trajetórias de Y(0) divergem antes do tratamento')
salvar_figura(fig, 'd19_03_violacao_parallel_trends')
fig.show()

,slope_y0_tratados,slope_y0_controles,diferenca_slope_pre,violacao_detectavel_no_dgp
0,1.960368,1.210368,0.75,True


Figura preservada em: outputs\figures\interactive\d19_03_violacao_parallel_trends.html


A divergência está no próprio outcome potencial sem tratamento. Nenhum estimador corrige automaticamente uma hipótese identificadora falsa; implementação correta de software e validade causal são dimensões distintas.

## 8. Overlap fraco

In [9]:
overlap_fraco = cenarios['OVERLAP_FRACO']
diagnostico_overlap = simulador.diagnosticar_overlap_dgp(overlap_fraco.painel)
display(pd.DataFrame([diagnostico_overlap]))
unidades_overlap = overlap_fraco.painel.drop_duplicates('id').assign(grupo=lambda d: np.where(d['coorte_g'].eq(0), 'Never-treated', 'Tratados'))
fig = px.histogram(unidades_overlap, x='propensity_score_dgp', color='grupo', barmode='overlay', opacity=0.65, nbins=30, labels={'propensity_score_dgp': 'Score conhecido do DGP', 'count': 'Unidades', 'grupo': 'Grupo'}, title='Cenário sintético de overlap fraco')
salvar_figura(fig, 'd19_04_overlap_fraco')
fig.show()

,score_min_tratados,score_max_tratados,score_min_controles,score_max_controles,n_tratados_acima_max_controles,n_tratados_abaixo_min_controles,fracao_tratados_no_suporte
0,0.767419,0.971776,0.024141,0.745996,144,0,0.0


Figura preservada em: outputs\figures\interactive\d19_04_overlap_fraco.html

O cenário produz deliberadamente tratados acima do máximo dos controles. Isso conecta o mecanismo aos cinco tratados extremos da D18, mas não usa os outcomes reais desses municípios e não executa restrição de suporte.

## 9. Coorte pequena: identificação versus precisão

In [10]:
coorte_pequena = cenarios['COORTE_2013_N2']
contagens_pequena = coorte_pequena.painel.drop_duplicates('id').query('coorte_g > 0').groupby('coorte_g', as_index=False).size().rename(columns={'size': 'n_tratados'})
mostrar(contagens_pequena, 10)
fig = px.bar(contagens_pequena, x='coorte_g', y='n_tratados', text='n_tratados', color_discrete_sequence=[CORES_IPT['AZUL_PRINCIPAL']], labels={'coorte_g': 'Coorte', 'n_tratados': 'Unidades tratadas'}, title='Coorte sintética n=2: baixa precisão não é falha automática de identificação')
fig.update_traces(textposition='outside')
salvar_figura(fig, 'd19_05_coorte_pequena')
fig.show()

,coorte_g,n_tratados
0,2010,71
1,2012,71
2,2013,2


Figura preservada em: outputs\figures\interactive\d19_05_coorte_pequena.html

A coorte `n=2` compartilha o mesmo desenho identificador do restante do DGP limpo, mas possui pouca informação para estimação e inferência. O paralelo correto com 2013 é risco de precisão e instabilidade, não exclusão automática.

## 10. Agregações da verdade do DGP

In [11]:
agregacao_grupo = simulador.agregar_verdade_att(heterogeneo.verdade_att_gt, 'grupo')
agregacao_dinamica = simulador.agregar_verdade_att(heterogeneo.verdade_att_gt, 'dinamica')
agregacao_global = simulador.agregar_verdade_att(heterogeneo.verdade_att_gt, 'global')
mostrar(agregacao_grupo, 10)
mostrar(agregacao_dinamica, 20)
mostrar(agregacao_global, 5)

,coorte_g,peso,att_verdadeiro_agregado,tipo_agregacao,natureza
0,2010,480,8.816653,grupo,VERDADE_DGP_NAO_ESTIMATIVA
1,2012,384,11.103649,grupo,VERDADE_DGP_NAO_ESTIMATIVA
2,2013,336,15.273349,grupo,VERDADE_DGP_NAO_ESTIMATIVA


,event_time,peso,att_verdadeiro_agregado,tipo_agregacao,natureza
0,0,144,5.314550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
1,1,144,7.064550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
2,2,144,8.814550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
3,3,144,10.564550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
4,4,144,12.314550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
5,5,144,14.064550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
6,6,144,15.814550,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
7,7,96,15.210151,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
8,8,48,14.941653,dinamica,VERDADE_DGP_NAO_ESTIMATIVA
9,9,48,16.691653,dinamica,VERDADE_DGP_NAO_ESTIMATIVA


,peso,att_verdadeiro_agregado,tipo_agregacao,natureza
0,1200,11.356367,global,VERDADE_DGP_NAO_ESTIMATIVA


As perguntas são distintas: `ATT(g,t)` localiza grupo e calendário; agregação por grupo resume cada coorte; agregação dinâmica resume tempo desde a exposição; agregação global resume todas as células elegíveis. Estas tabelas agregam a verdade conhecida, não resultados estimados.

## 11. Event-study sintético: leads, referência e lags

In [12]:
event_truth = heterogeneo.verdade_event_time.query('event_time >= -3 and event_time <= 4').copy()
fig = px.line(event_truth, x='event_time', y='efeito_verdadeiro', markers=True, labels={'event_time': 'k = t-g', 'efeito_verdadeiro': 'Efeito verdadeiro médio'}, title='Event-study da verdade sintética: k=-1 é referência de reporte')
fig.add_hline(y=0, line_dash='dash', line_color=CORES_IPT['AZUL_ESCURO'])
fig.add_vline(x=-1, line_dash='dot', line_color=CORES_IPT['CIANO'])
salvar_figura(fig, 'd19_06_event_study_verdade')
fig.show()

Figura preservada em: outputs\figures\interactive\d19_06_event_study_verdade.html


`k<0` são leads; `k=-1` é a referência; `k=0` é o primeiro período tratado; e `k>0` são lags. O gráfico mostra a verdade do DGP. O event-study estimado pelo backend moderno permanece bloqueado e não é substituído por TWFE ingênuo.

## 12. Inferência planejada para D20

In [13]:
inferencia_planejada = mapeamento.loc[mapeamento['conceito_artigo'].isin(['bootstrap', 'bandas simultâneas', 'iterações', 'cluster', 'nível', 'influence function'])].copy()
mostrar(inferencia_planejada, 20)

,conceito_artigo,parametro_backend,valor_projeto,justificativa
15,bootstrap,bstrap,TRUE,multiplier bootstrap
16,bandas simultâneas,cband,TRUE,inferência conjunta
17,iterações,biters,1000_MÍNIMO_A_CONFIRMAR_D20,não reduzir por conveniência
18,cluster,clustervars,codigo_municipio_ibge,cluster na unidade do painel
19,nível,alp,0.05,bandas de 95%
20,influence function,compute_inffunc,TRUE,necessária para inferência e aggte


Recomendação D20: multiplier bootstrap, bandas simultâneas de 95%, cluster municipal e influence functions preservadas para `aggte`. O mínimo documental é `biters=1000`; qualquer aumento deve ser definido por estabilidade/recursos antes de olhar efeitos reais. A configuração não foi executada porque o backend está ausente.

## 13. Dry-run estrutural da amostra real D15

In [14]:
dados_backend_real, validacao_real = infraestrutura.executar_dry_run_real(PATH_AMOSTRA)
validacao_real_export = pd.DataFrame([
    {'metrica': chave, 'valor': str(valor)} for chave, valor in validacao_real.items()
])
validacao_real_export.to_csv(OUT_DIAG / 'D19_dry_run_amostra_real.csv', index=False, encoding='utf-8-sig')
mostrar(validacao_real_export, 40)
assert validacao_real['estimacao_executada'] is False
assert validacao_real['byte_identical'] is True

,metrica,valor
0,n_linhas,66196
1,n_elegiveis,66195
2,n_unidades,5092
3,n_tratados,129
4,n_controles,4963
5,contagem_coortes,"{2009: 21, 2010: 27, 2011: 66, 2012: 13, 2013: 2}"
6,anos,"[2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]"
7,painel_base_balanceado,True
8,chave_unica,True
9,anos_inelegiveis_cabo_frio,[2009]


O dry-run preserva o painel inteiro e cria `g_backend=0` para never-treated. A máscara Cabo Frio/2009 continua explícita; o período de estimação não é truncado pela janela do event-study. `PREPARAR` e `ESTIMAR` permanecem separados por desenho.

## 14. Cinco tratados extremos e coorte 2013

In [15]:
codigos_extremos = {'2910800', '2604106', '5201108', '4305108', '1506807'}
extremos_preservados = dados_backend_real.loc[dados_backend_real['id_backend'].isin(codigos_extremos)].drop_duplicates('id_backend')[['id_backend', 'municipio_fonte', 'g_backend', 'papel_causal']].sort_values('id_backend')
assert set(extremos_preservados['id_backend']) == codigos_extremos
reporte_coortes = infraestrutura.resumo_coortes_para_reporte(dados_backend_real)
reporte_coortes.to_csv(OUT_DIAG / 'D19_coortes_reporte.csv', index=False, encoding='utf-8-sig')
mostrar(extremos_preservados, 10)
mostrar(reporte_coortes, 10)

,id_backend,municipio_fonte,g_backend,papel_causal
3016,1506807,Santarém - PA,2010,TRATADO_PRINCIPAL
17537,2604106,Caruaru - PE,2011,TRATADO_PRINCIPAL
22828,2910800,Feira de Santana - BA,2012,TRATADO_PRINCIPAL
55822,4305108,Caxias do Sul - RS,2011,TRATADO_PRINCIPAL
63427,5201108,Anápolis - GO,2011,TRATADO_PRINCIPAL


,coorte_g,n_tratados,warning_precisao,regra
0,2009,21,SEM_WARNING_CRITICO_N2,REPORTAR_SEM_EXCLUIR
1,2010,27,SEM_WARNING_CRITICO_N2,REPORTAR_SEM_EXCLUIR
2,2011,66,SEM_WARNING_CRITICO_N2,REPORTAR_SEM_EXCLUIR
3,2012,13,SEM_WARNING_CRITICO_N2,REPORTAR_SEM_EXCLUIR
4,2013,2,COORTE_N2_PRECISAO_CRITICA,REPORTAR_SEM_EXCLUIR


Os cinco tratados fora do suporte intervalar D18 e os dois tratados de 2013 permanecem no input. A infraestrutura apenas permite que uma futura D20 reporte sensibilidade de suporte e warning de precisão; nenhuma exclusão ou sensibilidade é disparada automaticamente.

## 15. Plano operacional D20

In [16]:
plano_d20 = infraestrutura.plano_operacional_d20()
plano_d20.to_csv(OUT_DIAG / 'D19_plano_operacional_D20.csv', index=False, encoding='utf-8-sig')
mostrar(plano_d20, 30)

,item,valor,justificativa,status
0,backend,R did,2.5.1,BLOQUEADO_DEPENDENCIA
1,funcao_principal,att_gt,backend oficial,NAO_EXECUTADO_D19
2,idname,codigo_municipio_ibge,contrato D14-D19,CONGELADO
3,tname,ano,2007-2019 integral,CONGELADO
4,gname,g_backend,0 para never-treated,CONGELADO
5,yname,pessoal_ocupado_assalariado,nível,CONGELADO
6,control_group,nevertreated,principal,CONGELADO
7,anticipation,0,hipótese identificadora,CONGELADO
8,base_period,universal,k=-1 normalizado,RECOMENDADO_D19
9,xformla,~1,principal incondicional,CONGELADO


O plano é operacional, mas não autoriza execução. Antes da D20 será necessária autorização explícita para disponibilizar um backend, reexecutar todos os cenários sintéticos e validar recuperação, bias, RMSE, cobertura, agregações e inferência. Somente depois dessa validação um novo gate poderá decidir se a amostra real pode ser estimada.

## 16. Gates e preservação final

In [17]:
gates = infraestrutura.gates_d19(decisao_backend['gate'])
assert infraestrutura.sha256_arquivo(PATH_AMOSTRA) == infraestrutura.HASH_D15_ESPERADO
assert validacao_real['n_tratados'] == 129
assert validacao_real['n_controles'] == 4963
assert validacao_real['anos_inelegiveis_cabo_frio'] == [2009]
assert validacao_real['estimacao_executada'] is False
display(pd.DataFrame({'gate': gates.keys(), 'valor': gates.values()}))

,gate,valor
0,D19_BACKEND_CALLAWAY_SANTANNA_VALIDADO,BLOQUEADO_DEPENDENCIA
1,D19_DGP_SINTETICO_VALIDADO,SIM
2,D19_ATT_GT_SINTETICO_VALIDADO,NAO
3,D19_HETEROGENEIDADE_SINTETICA_VALIDADA,SIM
4,D19_EVENT_STUDY_SINTETICO_VALIDADO,NAO
5,D19_INFERENCIA_CONFIGURADA,NAO
6,D19_DRY_RUN_AMOSTRA_REAL_VALIDADO,SIM
7,D19_INFRAESTRUTURA_PRONTA_PARA_ESTIMACAO_REAL,NAO
8,D19_EFEITO_CAUSAL_REAL_ESTIMADO,NAO
9,DESENHO_CAUSAL_APROVADO,NAO


## Takeaways

- O contrato e a interface de dados para Callaway–Sant'Anna estão definidos sem confundir painel de estimação com janela de reporte.
- O DGP e os diagnósticos sintéticos estão validados; TWFE é demonstrado somente em dados sintéticos e nunca chamado na D15.
- A validação específica de um backend `ATT(g,t)`, seu event-study e sua inferência não ocorreu porque nenhuma implementação adequada está instalada.
- A D15 passou pelo dry-run estrutural e permaneceu byte-identical.
- A D20 está especificada, mas bloqueada até uma decisão explícita de dependência e uma nova rodada de validação sintética.
- `D19_EFEITO_CAUSAL_REAL_ESTIMADO = NAO` e `DESENHO_CAUSAL_APROVADO = NAO`.